In [2]:
# Import des bibliothèques nécessaires au code.

import geopandas as gpd  #Analyse vecteur
from shapely.geometry import Point, Polygon #Analyse vecteur
from sklearn.preprocessing import MinMaxScaler #Analyse statistiques
import numpy as np  #Manipulation de données 
import pandas as pd  #Manipulation de données via des "tableaux"
import rasterio  #Analyse raster
from rasterio.warp import calculate_default_transform, reproject, Resampling  #Extensions de rasterio
import glob  #Traitement de fichier
import os  #Interaction avec le système d'exploitation
from rasterio.merge import merge  #Ajoute la fonction "merge" de rasterio afin de fusionner les tuiles rasters

In [3]:
# Transformation des couches shapefiles en GeoDataFrames, semblables à des tableaux avec géographie, utilisable par GéoPandas

Commune = gpd.read_file("C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/Avignon.shp")
Cadastre = gpd.read_file("C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/cadastre.shp")
Batiments = gpd.read_file("C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/batiments.shp")
Surface_hydro = gpd.read_file("C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/Surface_hydro.shp")
Troncon_route = gpd.read_file("C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/trancon_route.shp")

In [3]:
# Début de la fusion des tuiles RGE Alti 5m, chemin vers les fichiers d'origine et de destination

dossier_tuiles = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE\RGEALTI_2-0_5M_ASC_LAMB93-IGN69_D084_2022-12-16\RGEALTI\1_DONNEES_LIVRAISON_2023-01-00223\RGEALTI_MNT_5M_ASC_LAMB93_IGN69_D084/*.*"
sortie = r"C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/RGE/DME.tif"

In [4]:
# Fonction merge pour fusionner les tuiles RGE Alti 5m

mosaic, out_trans = merge(glob.glob(dossier_tuiles))

In [5]:
# Copie ("out_meta") les paramètres du premier fichier de dossier ('dossier_tuiles')

with rasterio.open(glob.glob(dossier_tuiles)[0]) as dem:
    out_meta = dem.meta.copy()

In [6]:
# Modifie les paramètres de "out_meta" : Format du fichier, hauteur, épaisseur et la géographie.

out_meta.update({
    'driver': 'GTiff',
    'height': mosaic.shape[1],
    'width': mosaic.shape[2],
    'transform': out_trans,
})

In [7]:
# Ecriture du fichier, format écriture

with rasterio.open(sortie, 'w', **out_meta) as dem:
    dem.write(mosaic)

In [8]:
# Début du calcul de la pente par GDAL, variable du fichier d'origine et de destination

dem_chemin = r"C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/RGE/DME.tif"
slope_chemin = r"C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/RGE/slope.tif"

In [ ]:
# Méthode numéro 1 : Calcul de la pente par l'API GDAL.  !! Ne marche pas sur mon code, sûrement
# par l'écriture du fichier qui malfonctionne. Nous allons donc calculer le code par une commande bash

#from osgeo import gdal  (Import de GDAL)

#dem = gdal.Open(dem_chemin) Fariable GDAL de "dem_chemin"

# options = gdal.DEMProcessingOptions(   #Création des options pour le calcul de la pente
    #format='GTiff',
    #slopeFormat='degree',
    #computeEdges=True,
    #zeroForFlat=True,
    #alg='Horn',
#)

#gdal.DEMProcessing(   #Calcul de la pentre avec "slope"
    #destName=slope_chemin,
    #srcDS=dem,
    #processing='slope',
    #options=options
#)


Fichier libéré et prêt à être utilisé.


In [1]:
# Utilise subprocess pour exécuter une commande bash

import subprocess

In [ ]:



# Ecriture du chemin des fichiers
dme_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE\DME.tif"
slope_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE\slope.tif"

# Commande GDAL avec l'algorithme "slope"
cmd = [
    "gdaldem",
    "slope",
    dme_path,
    slope_path,
    "-of", "GTiff",
    "-s", "1",
    "-alg", "Horn"
]

subprocess.run(cmd, check=True)

CompletedProcess(args=['gdaldem', 'slope', 'C:\\Users\\edwin\\Desktop\\QGIS TUTORIAL\\Schema_de_traitement_Python\\RGE\\DME.tif', 'C:\\Users\\edwin\\Desktop\\QGIS TUTORIAL\\Schema_de_traitement_Python\\RGE\\slope.tif', '-of', 'GTiff', '-s', '1', '-alg', 'Horn'], returncode=0)

In [3]:
# Création du fichier "calculated", un raster binaire avec 0 = Pente inférieure à 15 degrès, et 1= Pente supérieure à 15 degrès



slope_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/slope.tif"
calculated_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated.tif"


cmd = [
    "python",
    "-m", "osgeo_utils.gdal_calc",
    "-A", slope_path,
    "--outfile=" + calculated_path,
    "--calc=A>15",
    "--type=Byte",
    "--NoDataValue=0",
    "--overwrite"
]


subprocess.run(cmd, check=True)


CompletedProcess(args=['python', '-m', 'osgeo_utils.gdal_calc', '-A', 'C:\\Users\\edwin\\Desktop\\QGIS TUTORIAL\\Schema_de_traitement_Python\\RGE/slope.tif', '--outfile=C:\\Users\\edwin\\Desktop\\QGIS TUTORIAL\\Schema_de_traitement_Python\\RGE/calculated.tif', '--calc=A>15', '--type=Byte', '--NoDataValue=0', '--overwrite'], returncode=0)

In [ ]:
# La méthode de vectorisaion par GDAL. Fonctionne mais est très lente

#calculated_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated.tif"
##vectorized_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/vectorized.gpkg"

#cmd = [
    #"python",
    #r"C:\Users\edwin\miniconda3\envs\geo\Scripts\gdal_polygonize.py",
    #calculated_path,
    #"-f", "GPKG",
    #"-b", "1",
    #vectorized_path
#]

#subprocess.run(cmd, check=True)

In [ ]:
# Avant de vectoriser notre raster "calculated", il faut le reprojeter en 2154, avec rasterio

from rasterio.warp import calculate_default_transform, reproject, Resampling

# Chemins des fichiers
calculated_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated.tif"
calculated_2154_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated_2154.tif"


with rasterio.open(calculated_path) as src:
    raster = src.read(1)
    profile = src.profile

    # Définition du EPSG : 2154 
    profile.update(crs='EPSG:2154')

    # Calculer la transformation pour la reprojection
    transform, width, height = calculate_default_transform(
        src.crs, profile['crs'], src.width, src.height, *src.bounds
    )
    profile.update(transform=transform, width=width, height=height)

    # Créer et sauvegarder le raster reprojeté
    with rasterio.open(calculated_2154_path, 'w', **profile) as dst:
        reproject(
            source=raster,
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=2154,
            dst_transform=transform,
            dst_crs='EPSG:2154',
            resampling=Resampling.nearest
        )


Raster reprojeté en EPSG:2154 : C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated_2154.tif


In [ ]:
# Nous allons désormais effectuer la vectorisation avec rasterio. Puis le sauvegarder en gpkg


from rasterio.features import shapes


# Chemins des fichiers
raster_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated_2154.tif"
output_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/vectorized.gpkg"

# Charger le raster
with rasterio.open(raster_path) as src:
    raster = src.read(1)
    transform = src.transform
    crs = src.crs

# Polygoniser  avec la valeur 1
polygons = shapes(raster, mask=(raster == 1), transform=transform)

# Créer le GeoDataFrame
pente_gdf = gpd.GeoDataFrame.from_features(
    [{"properties": {"valeur": 1}, "geometry": geom} for geom, val in polygons],
    crs=src.crs
)


# Sauvegarder en GeoPackage
pente_gdf.to_file(output_path, driver="GPKG")

In [ ]:
Commune = Commune.to_crs("EPSG:2154")
Cadastre = Cadastre.to_crs("EPSG:2154")
Batiments = Batiments.to_crs("EPSG:2154")
Surface_hydro = Surface_hydro.to_crs("EPSG:2154")
Troncon_route = Troncon_route.to_crs("EPSG:2154")
pente_gdf = pente_gdf.to_crs("EPSG:2154")